In [ ]:
import pickle
import dagshub
import mlflow
from mlflow.tracking import MlflowClient

dagshub.init(repo_owner="karev23", repo_name="ML-HW-1", mlflow=True)

# get model info from Registry
client = MlflowClient()
model_info = client.get_model_version("house-prices-best-model", "3")
print(f"Model name: {model_info.name}")
print(f"Version: {model_info.version}")
print(f"Run ID: {model_info.run_id}")

# load model locally
with open("models/gradient_boosting_best.pkl", "rb") as f:
    model = pickle.load(f)

print("Model loaded!")

Initialized MLflow to track repo "karev23/ML-HW-1"

Repository karev23/ML-HW-1 initialized!

Model name: house-prices-best-model
Version: 2
Run ID: 867976603a40454183ac933e1aaebb17
Model loaded!


In [5]:
# load and preprocess test data
test = pd.read_csv("data/test.csv")
test_ids = test["Id"]

# drop same columns as training
cols_to_drop = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu", "Id"]
test = test.drop(columns=cols_to_drop)

# fill missing values
for col in test.select_dtypes(include='number').columns:
    test[col] = test[col].fillna(test[col].median())
for col in test.select_dtypes(include='object').columns:
    test[col] = test[col].fillna("None")

# ordinal encoding
quality_map = {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0, "NA": 0}
BsmtExposure_map = {"Gd": 4, "Av": 3, "Mn": 2, "No": 1, "None": 0, "NA": 0}
BsmtFinType_map  = {"GLQ": 6, "ALQ": 5, "BLQ": 4, "Rec": 3, "LwQ": 2, "Unf": 1, "None": 0, "NA": 0}
GarageFinish_map = {"Fin": 3, "RFn": 2, "Unf": 1, "None": 0, "NA": 0}
Functional_map   = {"Typ": 7, "Min1": 6, "Min2": 5, "Mod": 4, "Maj1": 3, "Maj2": 2, "Sev": 1, "Sal": 0, "None": 0}
PavedDrive_map   = {"Y": 2, "P": 1, "N": 0, "None": 0}
LandSlope_map    = {"Gtl": 2, "Mod": 1, "Sev": 0, "None": 0}

ordinal_quality_cols = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond",
                        "HeatingQC", "KitchenQual", "GarageQual", "GarageCond"]
for col in ordinal_quality_cols:
    test[col] = test[col].map(quality_map)

test["BsmtExposure"] = test["BsmtExposure"].map(BsmtExposure_map)
test["BsmtFinType1"] = test["BsmtFinType1"].map(BsmtFinType_map)
test["BsmtFinType2"] = test["BsmtFinType2"].map(BsmtFinType_map)
test["GarageFinish"] = test["GarageFinish"].map(GarageFinish_map)
test["Functional"]   = test["Functional"].map(Functional_map)
test["PavedDrive"]   = test["PavedDrive"].map(PavedDrive_map)
test["LandSlope"]    = test["LandSlope"].map(LandSlope_map)
test["CentralAir"]   = (test["CentralAir"] == "Y").astype(int)

# one hot encoding
nominal_cols = [
    "MSZoning", "Street", "LotShape", "LandContour", "Utilities",
    "LotConfig", "Neighborhood", "Condition1", "Condition2",
    "BldgType", "HouseStyle", "RoofStyle", "RoofMatl",
    "Exterior1st", "Exterior2nd", "MasVnrType",
    "Foundation", "Heating", "Electrical",
    "GarageType", "SaleType", "SaleCondition"
]
test = pd.get_dummies(test, columns=nominal_cols, drop_first=True)
test = test.astype({col: int for col in test.select_dtypes(bool).columns})

# new features
test["TotalSF"]      = test["TotalBsmtSF"] + test["1stFlrSF"] + test["2ndFlrSF"]
test["HouseAge"]     = test["YrSold"] - test["YearBuilt"]
test["RemodAge"]     = test["YrSold"] - test["YearRemodAdd"]
test["WasRemodeled"] = (test["YearBuilt"] != test["YearRemodAdd"]).astype(int)
test["TotalBath"]    = test["FullBath"] + 0.5*test["HalfBath"] + test["BsmtFullBath"] + 0.5*test["BsmtHalfBath"]
test["HasGarage"]    = (test["GarageArea"] > 0).astype(int)
test["HasPool"]      = (test["PoolArea"] > 0).astype(int)
test["HasFireplace"] = (test["Fireplaces"] > 0).astype(int)
test["HasBsmt"]      = (test["TotalBsmtSF"] > 0).astype(int)
test["TotalPorch"]   = test["OpenPorchSF"] + test["EnclosedPorch"] + test["3SsnPorch"] + test["ScreenPorch"] + test["WoodDeckSF"]
test["LotArea_log"]  = np.log1p(test["LotArea"])
test["GrLivArea_log"]= np.log1p(test["GrLivArea"])

# fill any remaining NaN
for col in test.columns:
    if test[col].isnull().sum() > 0:
        test[col] = test[col].fillna(0)

# align columns with training
with open("models/train_columns.pkl", "rb") as f:
    train_columns = pickle.load(f)
test = test.reindex(columns=train_columns, fill_value=0)


C:\Users\User\AppData\Local\Temp\ipykernel_9404\1378894398.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test["TotalSF"]      = test["TotalBsmtSF"] + test["1stFlrSF"] + test["2ndFlrSF"]
C:\Users\User\AppData\Local\Temp\ipykernel_9404\1378894398.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test["HouseAge"]     = test["YrSold"] - test["YearBuilt"]
C:\Users\User\AppData\Local\Temp\ipykernel_9404\1378894398.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert`

In [6]:

test_preds_log = model.predict(test)


test_preds = np.expm1(test_preds_log)


submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": test_preds
})

submission.to_csv("data/submission.csv", index=False)
